# 00 - Institutional Data Acquisition Plan

**Crypto Statistical Arbitrage: Multi-Venue Data Infrastructure**

This notebook documents the comprehensive data acquisition strategy for cross-venue
statistical arbitrage research, covering 44 data sources (32 enabled) across CEX, 
hybrid, DEX, and options venues.

**System Version:** 3.0.0  
**Phase 1 Status:** Complete

---

## Executive Summary

| Metric | Value |
|--------|-------|
| **Target Date Range** | Jan 2020 - Jan 2026 (6 years) |
| **Primary Assets** | BTC, ETH + 27 liquid alts (29 total) |
| **Venue Coverage** | 44 collectors (32 enabled) |
| **Data Types** | Funding rates, OHLCV, OI, DEX pools, options |
| **Detailed Analysis** | Survivorship bias, wash trading, MEV, cross-validation |
| **Quality Score** | 84.5/100 (GOOD) |

---

## How to Use This Notebook

This notebook is for **documentation purposes**. To actually collect data, run:

```bash
# Default comprehensive mode (recommended)
python run_phase1.py --start 2020-01-01 --end 2026-01-31

# The comprehensive pipeline includes:
# - Data collection from 32 venues
# - 7-stage cleaning pipeline
# - Survivorship bias assessment
# - Wash trading detection
# - MEV/Sandwich attack analysis
# - Cross-venue validation
```

---

## Table of Contents

1. [Strategy-Driven Data Requirements](#1-strategy-driven-data-requirements)
2. [Asset Universe Definition](#2-asset-universe-definition)
3. [Venue Evaluation Matrix](#3-venue-evaluation-matrix)
4. [Technical Collection Architecture](#4-technical-collection-architecture)
5. [Quality Assurance Framework](#5-quality-assurance-framework)
6. [Risk Assessment and Mitigation](#6-risk-assessment-and-mitigation)
7. [Execution Timeline](#7-execution-timeline)

In [1]:
"""
Data Acquisition Plan Configuration
====================================
Core parameters for the data collection infrastructure.
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Dict, List, Optional
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# PROJECT CONFIGURATION
# =============================================================================

@dataclass
class ProjectConfig:
    """Central configuration for data acquisition project."""
    
    # Date range
    start_date: str = '2022-01-01'
    end_date: str = '2024-12-31'
    
    # Quality targets
    min_coverage_pct: float = 95.0
    max_missing_pct: float = 5.0
    max_duplicate_pct: float = 1.0
    
    # Execution parameters
    max_concurrent_venues: int = 3
    rate_limit_buffer_pct: float = 20.0  # Stay 20% below rate limits
    
    @property
    def days_covered(self) -> int:
        start = datetime.strptime(self.start_date, '%Y-%m-%d')
        end = datetime.strptime(self.end_date, '%Y-%m-%d')
        return (end - start).days + 1
    
    @property
    def hours_covered(self) -> int:
        return self.days_covered * 24

CONFIG = ProjectConfig()

print(f"Project Date Range: {CONFIG.start_date} to {CONFIG.end_date}")
print(f"Coverage Period: {CONFIG.days_covered:,} days ({CONFIG.hours_covered:,} hours)")
print(f"Quality Target: >{CONFIG.min_coverage_pct}% coverage, <{CONFIG.max_missing_pct}% missing")

Project Date Range: 2022-01-01 to 2024-12-31
Coverage Period: 1,096 days (26,304 hours)
Quality Target: >95.0% coverage, <5.0% missing


---

## 1. Strategy-Driven Data Requirements

Data requirements are derived from the four core statistical arbitrage strategies.
Each strategy has specific data needs that drive venue selection and collection priorities.

In [2]:
class DataPriority(Enum):
    """Data priority levels for collection planning."""
    CRITICAL = 1    # Strategy cannot function without
    REQUIRED = 2    # Essential for full functionality
    IMPORTANT = 3   # Enhances strategy performance
    OPTIONAL = 4    # Nice-to-have, not essential

@dataclass
class DataRequirement:
    """Single data requirement specification."""
    strategy: str
    data_type: str
    frequency: str
    assets: str
    venues: str
    priority: DataPriority
    notes: str = ""
    
    @property
    def priority_label(self) -> str:
        return self.priority.name.title()

# Strategy 1: Cross-Venue Funding Rate Arbitrage
FUNDING_ARB_REQS = [
    DataRequirement(
        strategy='Cross-Venue Funding Arbitrage',
        data_type='Funding Rates',
        frequency='8h (CEX), 1h (hybrid)',
        assets='BTC, ETH, SOL + 10 liquid alts',
        venues='Binance, Bybit, Hyperliquid, dYdX',
        priority=DataPriority.CRITICAL,
        notes='Core signal - settlement times must be precise'
    ),
    DataRequirement(
        strategy='Cross-Venue Funding Arbitrage',
        data_type='Mark Prices',
        frequency='1h',
        assets='Same as funding',
        venues='All perp venues',
        priority=DataPriority.CRITICAL,
        notes='Required for position P&L calculation'
    ),
    DataRequirement(
        strategy='Cross-Venue Funding Arbitrage',
        data_type='Open Interest',
        frequency='4h',
        assets='BTC, ETH',
        venues='Binance, Bybit, Hyperliquid',
        priority=DataPriority.IMPORTANT,
        notes='Crowding indicator for capacity estimation'
    ),
    DataRequirement(
        strategy='Cross-Venue Funding Arbitrage',
        data_type='Spot Prices',
        frequency='1h',
        assets='Same as funding',
        venues='Binance spot',
        priority=DataPriority.REQUIRED,
        notes='Basis calculation reference'
    ),
]

# Strategy 2: Altcoin Pairs Trading
PAIRS_TRADING_REQS = [
    DataRequirement(
        strategy='Altcoin Pairs Trading',
        data_type='OHLCV',
        frequency='1h',
        assets='50+ altcoins (market cap filtered)',
        venues='Binance (primary), Uniswap V3 (DEX)',
        priority=DataPriority.CRITICAL,
        notes='Cointegration analysis requires long history'
    ),
    DataRequirement(
        strategy='Altcoin Pairs Trading',
        data_type='Market Capitalization',
        frequency='1d',
        assets='Full universe',
        venues='CoinGecko, CoinMarketCap',
        priority=DataPriority.REQUIRED,
        notes='Universe filtering and sector classification'
    ),
    DataRequirement(
        strategy='Altcoin Pairs Trading',
        data_type='DEX Pool TVL',
        frequency='1d',
        assets='DEX-traded tokens',
        venues='Uniswap V3 (The Graph)',
        priority=DataPriority.REQUIRED,
        notes='Liquidity filter for DEX execution'
    ),
    DataRequirement(
        strategy='Altcoin Pairs Trading',
        data_type='Trading Volume',
        frequency='1d',
        assets='Full universe',
        venues='CoinGecko aggregated',
        priority=DataPriority.IMPORTANT,
        notes='Liquidity screening'
    ),
]

# Strategy 3: BTC Futures Term Structure
TERM_STRUCTURE_REQS = [
    DataRequirement(
        strategy='BTC Futures Term Structure',
        data_type='Quarterly Futures Prices',
        frequency='1h',
        assets='BTC (all expirations)',
        venues='Binance, Deribit, CME',
        priority=DataPriority.CRITICAL,
        notes='Multiple expiries required for curve construction'
    ),
    DataRequirement(
        strategy='BTC Futures Term Structure',
        data_type='Perpetual Prices + Funding',
        frequency='1h',
        assets='BTC',
        venues='Binance, Hyperliquid, dYdX',
        priority=DataPriority.CRITICAL,
        notes='Perp as front-month proxy'
    ),
    DataRequirement(
        strategy='BTC Futures Term Structure',
        data_type='Spot Price',
        frequency='1h',
        assets='BTC',
        venues='Binance spot, Coinbase',
        priority=DataPriority.CRITICAL,
        notes='Basis calculation denominator'
    ),
    DataRequirement(
        strategy='BTC Futures Term Structure',
        data_type='Interest Rates (risk-free)',
        frequency='1d',
        assets='USD rates',
        venues='FRED API / Treasury.gov',
        priority=DataPriority.IMPORTANT,
        notes='Fair value calculation'
    ),
]

# Strategy 4: Volatility Surface Arbitrage
VOL_SURFACE_REQS = [
    DataRequirement(
        strategy='Volatility Surface Arbitrage',
        data_type='Options Chain (full)',
        frequency='1h',
        assets='BTC, ETH',
        venues='Deribit',
        priority=DataPriority.CRITICAL,
        notes='All strikes and expiries for surface construction'
    ),
    DataRequirement(
        strategy='Volatility Surface Arbitrage',
        data_type='DVOL Index',
        frequency='1h',
        assets='BTC, ETH',
        venues='Deribit',
        priority=DataPriority.REQUIRED,
        notes='Benchmark implied volatility'
    ),
    DataRequirement(
        strategy='Volatility Surface Arbitrage',
        data_type='Underlying Spot/Perp',
        frequency='1h',
        assets='BTC, ETH',
        venues='Deribit, Binance',
        priority=DataPriority.CRITICAL,
        notes='Delta hedging reference'
    ),
    DataRequirement(
        strategy='Volatility Surface Arbitrage',
        data_type='Realized Volatility',
        frequency='1h',
        assets='BTC, ETH',
        venues='Derived from spot OHLCV',
        priority=DataPriority.REQUIRED,
        notes='Vol premium/discount calculation'
    ),
]

# Combine all requirements
ALL_REQUIREMENTS = (
    FUNDING_ARB_REQS + 
    PAIRS_TRADING_REQS + 
    TERM_STRUCTURE_REQS + 
    VOL_SURFACE_REQS
)

# Convert to DataFrame for display
requirements_df = pd.DataFrame([
    {
        'Strategy': r.strategy,
        'Data Type': r.data_type,
        'Frequency': r.frequency,
        'Assets': r.assets,
        'Venues': r.venues,
        'Priority': r.priority_label,
        'Notes': r.notes
    }
    for r in ALL_REQUIREMENTS
])

print(f"Total Data Requirements: {len(ALL_REQUIREMENTS)}")
print(f"\nBy Priority:")
print(requirements_df.groupby('Priority').size().sort_index())

requirements_df

Total Data Requirements: 16

By Priority:
Priority
Critical     8
Important    3
Required     5
dtype: int64


,Strategy,Data Type,Frequency,Assets,Venues,Priority,Notes
0,Cross-Venue Funding Arbitrage,Funding Rates,"8h (CEX), 1h (hybrid)","BTC, ETH, SOL + 10 liquid alts","Binance, Bybit, Hyperliquid, dYdX",Critical,Core signal - settlement times must be precise
1,Cross-Venue Funding Arbitrage,Mark Prices,1h,Same as funding,All perp venues,Critical,Required for position P&L calculation
2,Cross-Venue Funding Arbitrage,Open Interest,4h,"BTC, ETH","Binance, Bybit, Hyperliquid",Important,Crowding indicator for capacity estimation
3,Cross-Venue Funding Arbitrage,Spot Prices,1h,Same as funding,Binance spot,Required,Basis calculation reference
4,Altcoin Pairs Trading,OHLCV,1h,50+ altcoins (market cap filtered),"Binance (primary), Uniswap V3 (DEX)",Critical,Cointegration analysis requires long history
5,Altcoin Pairs Trading,Market Capitalization,1d,Full universe,"CoinGecko, CoinMarketCap",Required,Universe filtering and sector classification
6,Altcoin Pairs Trading,DEX Pool TVL,1d,DEX-traded tokens,Uniswap V3 (The Graph),Required,Liquidity filter for DEX execution
7,Altcoin Pairs Trading,Trading Volume,1d,Full universe,CoinGecko aggregated,Important,Liquidity screening
8,BTC Futures Term Structure,Quarterly Futures Prices,1h,BTC (all expirations),"Binance, Deribit, CME",Critical,Multiple expiries required for curve construction
9,BTC Futures Term Structure,Perpetual Prices + Funding,1h,BTC,"Binance, Hyperliquid, dYdX",Critical,Perp as front-month proxy


---

## 2. Asset Universe Definition

The asset universe is tiered by liquidity, market cap, and strategic importance.
Each tier has different data coverage requirements.

In [3]:
@dataclass
class AssetTier:
    """Asset tier definition with coverage requirements."""
    name: str
    symbols: List[str]
    min_coverage_pct: float
    data_types: List[str]
    description: str
    
    @property
    def count(self) -> int:
        return len(self.symbols)

ASSET_TIERS = [
    AssetTier(
        name='Tier 1: Core Assets',
        symbols=['BTC', 'ETH'],
        min_coverage_pct=99.0,
        data_types=['funding', 'ohlcv', 'open_interest', 'options', 'futures_curve'],
        description='Maximum data depth required - all strategies'
    ),
    AssetTier(
        name='Tier 2: Major Altcoins',
        symbols=['SOL', 'AVAX', 'ARB', 'OP', 'MATIC', 'LINK', 'DOGE', 'XRP'],
        min_coverage_pct=97.0,
        data_types=['funding', 'ohlcv', 'open_interest'],
        description='High liquidity alts - funding arb + pairs trading'
    ),
    AssetTier(
        name='Tier 3: DeFi Tokens',
        symbols=['UNI', 'AAVE', 'GMX', 'DYDX', 'CRV', 'LDO', 'MKR', 'SNX', 'COMP', '1INCH'],
        min_coverage_pct=95.0,
        data_types=['funding', 'ohlcv', 'dex_pools'],
        description='DeFi sector - pairs trading + DEX liquidity'
    ),
    AssetTier(
        name='Tier 4: Layer 1/2',
        symbols=['NEAR', 'ATOM', 'DOT', 'FTM', 'IMX', 'APT', 'SUI', 'SEI', 'INJ', 'TIA'],
        min_coverage_pct=93.0,
        data_types=['funding', 'ohlcv'],
        description='L1/L2 ecosystem - pairs trading'
    ),
    AssetTier(
        name='Tier 5: Thematic',
        symbols=['AXS', 'SAND', 'MANA', 'GALA', 'RNDR', 'FET', 'AGIX', 'WLD'],
        min_coverage_pct=90.0,
        data_types=['ohlcv'],
        description='Gaming/AI/Metaverse - thematic pairs'
    ),
]

# Display universe summary
print("=" * 70)
print("ASSET UNIVERSE DEFINITION")
print("=" * 70)

total_symbols = 0
for tier in ASSET_TIERS:
    print(f"\n{tier.name} ({tier.count} assets)")
    print(f"  Coverage target: {tier.min_coverage_pct}%")
    print(f"  Data types: {', '.join(tier.data_types)}")
    print(f"  Symbols: {', '.join(tier.symbols)}")
    total_symbols += tier.count

print(f"\n{'=' * 70}")
print(f"TOTAL UNIVERSE: {total_symbols} assets")
print("=" * 70)

ASSET UNIVERSE DEFINITION

Tier 1: Core Assets (2 assets)
  Coverage target: 99.0%
  Data types: funding, ohlcv, open_interest, options, futures_curve
  Symbols: BTC, ETH

Tier 2: Major Altcoins (8 assets)
  Coverage target: 97.0%
  Data types: funding, ohlcv, open_interest
  Symbols: SOL, AVAX, ARB, OP, MATIC, LINK, DOGE, XRP

Tier 3: DeFi Tokens (10 assets)
  Coverage target: 95.0%
  Data types: funding, ohlcv, dex_pools
  Symbols: UNI, AAVE, GMX, DYDX, CRV, LDO, MKR, SNX, COMP, 1INCH

Tier 4: Layer 1/2 (10 assets)
  Coverage target: 93.0%
  Data types: funding, ohlcv
  Symbols: NEAR, ATOM, DOT, FTM, IMX, APT, SUI, SEI, INJ, TIA

Tier 5: Thematic (8 assets)
  Coverage target: 90.0%
  Data types: ohlcv
  Symbols: AXS, SAND, MANA, GALA, RNDR, FET, AGIX, WLD

TOTAL UNIVERSE: 38 assets


In [4]:
# Universe summary table
universe_summary = pd.DataFrame([
    {
        'Tier': tier.name.split(':')[0],
        'Category': tier.name.split(':')[1].strip() if ':' in tier.name else tier.name,
        'Assets': tier.count,
        'Coverage Target': f"{tier.min_coverage_pct}%",
        'Data Types': len(tier.data_types),
        'Symbols': ', '.join(tier.symbols[:5]) + ('...' if tier.count > 5 else '')
    }
    for tier in ASSET_TIERS
])

universe_summary

,Tier,Category,Assets,Coverage Target,Data Types,Symbols
0,Tier 1,Core Assets,2,99.0%,5,"BTC, ETH"
1,Tier 2,Major Altcoins,8,97.0%,3,"SOL, AVAX, ARB, OP, MATIC..."
2,Tier 3,DeFi Tokens,10,95.0%,3,"UNI, AAVE, GMX, DYDX, CRV..."
3,Tier 4,Layer 1/2,10,93.0%,2,"NEAR, ATOM, DOT, FTM, IMX..."
4,Tier 5,Thematic,8,90.0%,1,"AXS, SAND, MANA, GALA, RNDR..."


---

## 3. Venue Evaluation Matrix

Detailed evaluation of data sources by venue type, considering:
- Data coverage and quality
- API accessibility and rate limits
- Cost and authentication requirements
- Reliability and uptime

In [5]:
@dataclass
class VenueEvaluation:
    """Venue evaluation for data collection planning."""
    venue: str
    venue_type: str
    data_types: List[str]
    coverage: str
    cost: str
    rate_limit: str
    quality_score: int  # 1-10
    reliability_score: int  # 1-10
    decision: str
    notes: str = ""
    
    @property
    def overall_score(self) -> float:
        return (self.quality_score + self.reliability_score) / 2

# CEX Venue Evaluations
CEX_VENUES = [
    VenueEvaluation(
        venue='Binance',
        venue_type='CEX',
        data_types=['funding', 'ohlcv', 'open_interest', 'trades', 'orderbook'],
        coverage='500+ perps, 1500+ spot pairs',
        cost='Free (public endpoints)',
        rate_limit='1200 req/min (weighted)',
        quality_score=10,
        reliability_score=9,
        decision='PRIMARY',
        notes='Best coverage, highest rate limits. 8h funding interval.'
    ),
    VenueEvaluation(
        venue='Bybit',
        venue_type='CEX',
        data_types=['funding', 'ohlcv', 'open_interest'],
        coverage='300+ perps',
        cost='Free',
        rate_limit='120 req/min',
        quality_score=8,
        reliability_score=8,
        decision='VALIDATION',
        notes='Cross-validation source for funding rates.'
    ),
    VenueEvaluation(
        venue='OKX',
        venue_type='CEX',
        data_types=['funding', 'ohlcv', 'open_interest'],
        coverage='200+ perps',
        cost='Free',
        rate_limit='60 req/min',
        quality_score=8,
        reliability_score=7,
        decision='BACKUP',
        notes='Lower rate limits, useful for validation.'
    ),
    VenueEvaluation(
        venue='Deribit',
        venue_type='CEX (Options)',
        data_types=['options_chain', 'dvol', 'funding', 'ohlcv', 'futures'],
        coverage='BTC, ETH only',
        cost='Free',
        rate_limit='20 req/sec',
        quality_score=10,
        reliability_score=9,
        decision='PRIMARY OPTIONS',
        notes='Dominant options venue. Essential for vol surface strategy.'
    ),
]

# Hybrid Venue Evaluations
HYBRID_VENUES = [
    VenueEvaluation(
        venue='Hyperliquid',
        venue_type='Hybrid',
        data_types=['funding', 'ohlcv', 'open_interest', 'trades'],
        coverage='50+ perps',
        cost='Free',
        rate_limit='100 req/min',
        quality_score=9,
        reliability_score=8,
        decision='PRIMARY HYBRID',
        notes='1h funding interval. Best hybrid liquidity. Zero maker fees.'
    ),
    VenueEvaluation(
        venue='dYdX V4',
        venue_type='Hybrid',
        data_types=['funding', 'ohlcv', 'open_interest'],
        coverage='40+ perps',
        cost='Free',
        rate_limit='100 req/min',
        quality_score=8,
        reliability_score=8,
        decision='SECONDARY HYBRID',
        notes='1h funding. Cosmos-based, different risk profile.'
    ),
    VenueEvaluation(
        venue='GMX V2',
        venue_type='Hybrid (DEX)',
        data_types=['funding', 'ohlcv', 'open_interest'],
        coverage='20+ perps',
        cost='Free',
        rate_limit='GraphQL limited',
        quality_score=7,
        reliability_score=7,
        decision='MONITORING',
        notes='Variable funding. GLP-based liquidity.'
    ),
]

# DEX Venue Evaluations
DEX_VENUES = [
    VenueEvaluation(
        venue='Uniswap V3',
        venue_type='DEX',
        data_types=['pools', 'swaps', 'tvl', 'ohlcv'],
        coverage='1000s of pools (Ethereum, Arbitrum, Polygon)',
        cost='Free (100k queries/mo via The Graph)',
        rate_limit='Varies by indexer',
        quality_score=8,
        reliability_score=8,
        decision='PRIMARY DEX',
        notes='Best DEX liquidity. Requires wash trading filter.'
    ),
    VenueEvaluation(
        venue='GeckoTerminal',
        venue_type='DEX Aggregator',
        data_types=['pools', 'ohlcv', 'tvl'],
        coverage='Multi-chain',
        cost='Free (rate limited)',
        rate_limit='30 req/min',
        quality_score=7,
        reliability_score=7,
        decision='VALIDATION',
        notes='Good for cross-chain pool discovery.'
    ),
    VenueEvaluation(
        venue='DefiLlama',
        venue_type='Aggregator',
        data_types=['tvl', 'yields', 'protocol_metrics'],
        coverage='All major protocols',
        cost='Free',
        rate_limit='60 req/min',
        quality_score=9,
        reliability_score=9,
        decision='PRIMARY TVL',
        notes='Best TVL data source. Community maintained.'
    ),
]

# Combine all venues
ALL_VENUES = CEX_VENUES + HYBRID_VENUES + DEX_VENUES

# Create evaluation matrix
venue_matrix = pd.DataFrame([
    {
        'Venue': v.venue,
        'Type': v.venue_type,
        'Data Types': ', '.join(v.data_types),
        'Coverage': v.coverage,
        'Cost': v.cost,
        'Rate Limit': v.rate_limit,
        'Quality': v.quality_score,
        'Reliability': v.reliability_score,
        'Overall': v.overall_score,
        'Decision': v.decision
    }
    for v in ALL_VENUES
])

print("=" * 80)
print("VENUE EVALUATION MATRIX")
print("=" * 80)

venue_matrix.sort_values('Overall', ascending=False)

VENUE EVALUATION MATRIX


,Venue,Type,Data Types,Coverage,Cost,Rate Limit,Quality,Reliability,Overall,Decision
0,Binance,CEX,"funding, ohlcv, open_interest, trades, orderbook","500+ perps, 1500+ spot pairs",Free (public endpoints),1200 req/min (weighted),10,9,9.5,PRIMARY
3,Deribit,CEX (Options),"options_chain, dvol, funding, ohlcv, futures","BTC, ETH only",Free,20 req/sec,10,9,9.5,PRIMARY OPTIONS
9,DefiLlama,Aggregator,"tvl, yields, protocol_metrics",All major protocols,Free,60 req/min,9,9,9.0,PRIMARY TVL
4,Hyperliquid,Hybrid,"funding, ohlcv, open_interest, trades",50+ perps,Free,100 req/min,9,8,8.5,PRIMARY HYBRID
1,Bybit,CEX,"funding, ohlcv, open_interest",300+ perps,Free,120 req/min,8,8,8.0,VALIDATION
5,dYdX V4,Hybrid,"funding, ohlcv, open_interest",40+ perps,Free,100 req/min,8,8,8.0,SECONDARY HYBRID
7,Uniswap V3,DEX,"pools, swaps, tvl, ohlcv","1000s of pools (Ethereum, Arbitrum, Polygon)",Free (100k queries/mo via The Graph),Varies by indexer,8,8,8.0,PRIMARY DEX
2,OKX,CEX,"funding, ohlcv, open_interest",200+ perps,Free,60 req/min,8,7,7.5,BACKUP
6,GMX V2,Hybrid (DEX),"funding, ohlcv, open_interest",20+ perps,Free,GraphQL limited,7,7,7.0,MONITORING
8,GeckoTerminal,DEX Aggregator,"pools, ohlcv, tvl",Multi-chain,Free (rate limited),30 req/min,7,7,7.0,VALIDATION


In [6]:
# CEX vs Hybrid vs DEX Comparison
comparison_factors = pd.DataFrame([
    {
        'Factor': 'Liquidity Depth',
        'CEX': 'Excellent ($50M-500M daily)',
        'Hybrid': 'Good ($5M-50M daily)',
        'DEX': 'Variable ($100K-10M TVL)',
        'Arbitrage Impact': 'Determines position sizing'
    },
    {
        'Factor': 'Data Quality',
        'CEX': 'Excellent (clean orderbook)',
        'Hybrid': 'Good (on-chain verifiable)',
        'DEX': 'Requires filtering (wash trading)',
        'Arbitrage Impact': 'Signal reliability'
    },
    {
        'Factor': 'Funding Interval',
        'CEX': '8 hours (Binance, Bybit)',
        'Hybrid': '1 hour (HL, dYdX)',
        'DEX': 'Variable/continuous',
        'Arbitrage Impact': 'Trade frequency, normalization'
    },
    {
        'Factor': 'Transaction Costs',
        'CEX': 'Low (0.02-0.05%)',
        'Hybrid': 'Very Low (0-0.025%)',
        'DEX': 'Higher (0.3-1.0%)',
        'Arbitrage Impact': 'Breakeven spread calculation'
    },
    {
        'Factor': 'Counterparty Risk',
        'CEX': 'Exchange custody risk',
        'Hybrid': 'Smart contract risk',
        'DEX': 'Smart contract + IL risk',
        'Arbitrage Impact': 'Risk management'
    },
    {
        'Factor': 'API Accessibility',
        'CEX': 'REST + WebSocket',
        'Hybrid': 'REST + WebSocket',
        'DEX': 'Subgraph/RPC (slower)',
        'Arbitrage Impact': 'Collection efficiency'
    },
    {
        'Factor': 'Historical Data',
        'CEX': '3+ years available',
        'Hybrid': '1-2 years (newer venues)',
        'DEX': '2-3 years (varies)',
        'Arbitrage Impact': 'Backtest depth'
    },
])

print("\n" + "=" * 80)
print("VENUE TYPE COMPARISON")
print("=" * 80 + "\n")

comparison_factors


VENUE TYPE COMPARISON



,Factor,CEX,Hybrid,DEX,Arbitrage Impact
0,Liquidity Depth,Excellent ($50M-500M daily),Good ($5M-50M daily),Variable ($100K-10M TVL),Determines position sizing
1,Data Quality,Excellent (clean orderbook),Good (on-chain verifiable),Requires filtering (wash trading),Signal reliability
2,Funding Interval,"8 hours (Binance, Bybit)","1 hour (HL, dYdX)",Variable/continuous,"Trade frequency, normalization"
3,Transaction Costs,Low (0.02-0.05%),Very Low (0-0.025%),Higher (0.3-1.0%),Breakeven spread calculation
4,Counterparty Risk,Exchange custody risk,Smart contract risk,Smart contract + IL risk,Risk management
5,API Accessibility,REST + WebSocket,REST + WebSocket,Subgraph/RPC (slower),Collection efficiency
6,Historical Data,3+ years available,1-2 years (newer venues),2-3 years (varies),Backtest depth


---

## 4. Technical Collection Architecture

Data collection infrastructure design including parallelization strategy,
storage format, and estimated resource requirements.

In [7]:
@dataclass
class DataVolumeEstimate:
    """Estimated data volume for planning."""
    data_type: str
    records_per_day: int
    bytes_per_record: int
    compression_ratio: float = 0.3  # Parquet compression
    
    def total_records(self, days: int) -> int:
        return self.records_per_day * days
    
    def raw_size_mb(self, days: int) -> float:
        return (self.total_records(days) * self.bytes_per_record) / (1024 * 1024)
    
    def compressed_size_mb(self, days: int) -> float:
        return self.raw_size_mb(days) * self.compression_ratio

# Volume estimates
VOLUME_ESTIMATES = [
    DataVolumeEstimate(
        data_type='Funding Rates (all venues)',
        records_per_day=13 * 3 + 24 * 40,  # 3 venues @ 8h, 40 symbols @ 1h
        bytes_per_record=120
    ),
    DataVolumeEstimate(
        data_type='OHLCV 1h (CEX)',
        records_per_day=24 * 40 * 2,  # 40 symbols, 2 venues
        bytes_per_record=100
    ),
    DataVolumeEstimate(
        data_type='OHLCV 1h (Hybrid)',
        records_per_day=24 * 40 * 2,  # 40 symbols, 2 venues
        bytes_per_record=100
    ),
    DataVolumeEstimate(
        data_type='Open Interest',
        records_per_day=24 * 20 * 3,  # 20 symbols, 3 venues
        bytes_per_record=80
    ),
    DataVolumeEstimate(
        data_type='DEX Pools (Uniswap)',
        records_per_day=500,  # Top 500 pools daily
        bytes_per_record=200
    ),
    DataVolumeEstimate(
        data_type='Options Chain (Deribit)',
        records_per_day=24 * 500,  # ~500 options hourly
        bytes_per_record=150
    ),
    DataVolumeEstimate(
        data_type='BTC Futures Term Structure',
        records_per_day=24 * 6,  # 6 expiries hourly
        bytes_per_record=120
    ),
]

# Calculate totals
days = CONFIG.days_covered
volume_summary = pd.DataFrame([
    {
        'Data Type': v.data_type,
        'Records/Day': f"{v.records_per_day:,}",
        'Total Records': f"{v.total_records(days):,}",
        'Raw Size (MB)': f"{v.raw_size_mb(days):.1f}",
        'Compressed (MB)': f"{v.compressed_size_mb(days):.1f}"
    }
    for v in VOLUME_ESTIMATES
])

total_compressed = sum(v.compressed_size_mb(days) for v in VOLUME_ESTIMATES)

print("=" * 80)
print(f"DATA VOLUME ESTIMATES ({days:,} days)")
print("=" * 80 + "\n")

display(volume_summary)

print(f"\nTOTAL COMPRESSED STORAGE: {total_compressed:,.0f} MB ({total_compressed/1024:.2f} GB)")

DATA VOLUME ESTIMATES (1,096 days)



,Data Type,Records/Day,Total Records,Raw Size (MB),Compressed (MB)
0,Funding Rates (all venues),999,"1,094,904",125.3,37.6
1,OHLCV 1h (CEX),"1,920","2,104,320",200.7,60.2
2,OHLCV 1h (Hybrid),"1,920","2,104,320",200.7,60.2
3,Open Interest,"1,440","1,578,240",120.4,36.1
4,DEX Pools (Uniswap),500,"548,000",104.5,31.4
5,Options Chain (Deribit),"12,000","13,152,000",1881.4,564.4
6,BTC Futures Term Structure,144,"157,824",18.1,5.4



TOTAL COMPRESSED STORAGE: 795 MB (0.78 GB)


In [8]:
# Collection approach by data type
collection_architecture = pd.DataFrame([
    {
        'Data Type': 'Funding Rates',
        'Primary Source': 'Binance, Hyperliquid',
        'Method': 'REST API (paginated)',
        'Parallelization': 'Per-symbol async',
        'Storage': 'Parquet (monthly partitions)',
        'Validation': 'Cross-venue correlation',
        'Est. Time': '4-6 hours'
    },
    {
        'Data Type': 'OHLCV',
        'Primary Source': 'Binance, Hyperliquid',
        'Method': 'REST API (batch)',
        'Parallelization': 'Per-symbol async',
        'Storage': 'Parquet (symbol + month)',
        'Validation': 'OHLC consistency checks',
        'Est. Time': '6-8 hours'
    },
    {
        'Data Type': 'Open Interest',
        'Primary Source': 'Binance',
        'Method': 'REST API',
        'Parallelization': 'Per-symbol async',
        'Storage': 'Parquet (by venue)',
        'Validation': 'Range checks',
        'Est. Time': '2-3 hours'
    },
    {
        'Data Type': 'DEX Pools',
        'Primary Source': 'Uniswap (The Graph)',
        'Method': 'GraphQL queries',
        'Parallelization': 'Per-chain sequential',
        'Storage': 'Parquet (by chain)',
        'Validation': 'TVL > $500k filter',
        'Est. Time': '4-6 hours'
    },
    {
        'Data Type': 'Options Chain',
        'Primary Source': 'Deribit',
        'Method': 'REST API',
        'Parallelization': 'Sequential (rate limited)',
        'Storage': 'Parquet (by date)',
        'Validation': 'Greeks sanity checks',
        'Est. Time': '12-24 hours'
    },
    {
        'Data Type': 'Futures Term Structure',
        'Primary Source': 'Binance, Deribit',
        'Method': 'REST API',
        'Parallelization': 'Per-expiry async',
        'Storage': 'Parquet (by instrument)',
        'Validation': 'Contango/backwardation logic',
        'Est. Time': '2-3 hours'
    },
])

print("\n" + "=" * 80)
print("COLLECTION ARCHITECTURE")
print("=" * 80 + "\n")

collection_architecture


COLLECTION ARCHITECTURE



,Data Type,Primary Source,Method,Parallelization,Storage,Validation,Est. Time
0,Funding Rates,"Binance, Hyperliquid",REST API (paginated),Per-symbol async,Parquet (monthly partitions),Cross-venue correlation,4-6 hours
1,OHLCV,"Binance, Hyperliquid",REST API (batch),Per-symbol async,Parquet (symbol + month),OHLC consistency checks,6-8 hours
2,Open Interest,Binance,REST API,Per-symbol async,Parquet (by venue),Range checks,2-3 hours
3,DEX Pools,Uniswap (The Graph),GraphQL queries,Per-chain sequential,Parquet (by chain),TVL > $500k filter,4-6 hours
4,Options Chain,Deribit,REST API,Sequential (rate limited),Parquet (by date),Greeks sanity checks,12-24 hours
5,Futures Term Structure,"Binance, Deribit",REST API,Per-expiry async,Parquet (by instrument),Contango/backwardation logic,2-3 hours


---

## 5. Quality Assurance Framework

Thorough quality checks to ensure data integrity for trading research.

In [9]:
# Quality assurance checklist with categories
QA_FRAMEWORK = {
    'Coverage Validation': [
        ('Date range spans 2022-01-01 to 2024-12-31', 'CRITICAL'),
        ('Tier 1 assets (BTC, ETH) have >99% coverage', 'CRITICAL'),
        ('Tier 2 assets have >97% coverage', 'REQUIRED'),
        ('All funding settlement times captured', 'CRITICAL'),
        ('No gaps >24 hours in core data', 'REQUIRED'),
    ],
    'Cross-Validation': [
        ('Funding rates: Binance vs Bybit correlation >0.95', 'CRITICAL'),
        ('OHLCV: Cross-venue price deviation <0.5%', 'REQUIRED'),
        ('Open interest: Trend alignment across venues', 'IMPORTANT'),
        ('Options: Put-call parity within bounds', 'CRITICAL'),
    ],
    'Outlier Detection': [
        ('Funding rate outliers flagged (|rate| > 1%)', 'REQUIRED'),
        ('Price spikes >10% in 1h flagged', 'REQUIRED'),
        ('Volume anomalies detected (>5x average)', 'IMPORTANT'),
        ('Options IV outliers flagged (>300%)', 'REQUIRED'),
    ],
    'DEX-Specific': [
        ('Wash trading filter applied (volume/TVL ratio)', 'CRITICAL'),
        ('Minimum TVL threshold ($500k)', 'REQUIRED'),
        ('Pool age requirement (>30 days)', 'IMPORTANT'),
        ('Transaction count minimum (>100/day)', 'IMPORTANT'),
    ],
    'Documentation': [
        ('Data dictionary complete', 'REQUIRED'),
        ('Source attribution for all data', 'REQUIRED'),
        ('Known limitations documented', 'REQUIRED'),
        ('Survivorship bias assessment', 'IMPORTANT'),
        ('Delisted symbols tracked', 'IMPORTANT'),
    ],
    'Technical': [
        ('No hardcoded API keys', 'CRITICAL'),
        ('Collection reproducible from checkpoint', 'REQUIRED'),
        ('Timestamps normalized to UTC', 'CRITICAL'),
        ('Data types validated (funding as decimal)', 'CRITICAL'),
    ],
}

print("=" * 80)
print("QUALITY ASSURANCE FRAMEWORK")
print("=" * 80)

for category, checks in QA_FRAMEWORK.items():
    print(f"\n{category}:")
    print("-" * 60)
    for check, priority in checks:
        status = '[ ]'
        print(f"  {status} [{priority:8s}] {check}")

# Count totals
total_checks = sum(len(checks) for checks in QA_FRAMEWORK.values())
critical_checks = sum(1 for checks in QA_FRAMEWORK.values() for _, p in checks if p == 'CRITICAL')

print(f"\n{'=' * 80}")
print(f"TOTAL: {total_checks} checks ({critical_checks} CRITICAL)")
print("=" * 80)

QUALITY ASSURANCE FRAMEWORK

Coverage Validation:
------------------------------------------------------------
  [ ] [CRITICAL] Date range spans 2022-01-01 to 2024-12-31
  [ ] [CRITICAL] Tier 1 assets (BTC, ETH) have >99% coverage
  [ ] [REQUIRED] Tier 2 assets have >97% coverage
  [ ] [CRITICAL] All funding settlement times captured
  [ ] [REQUIRED] No gaps >24 hours in core data

Cross-Validation:
------------------------------------------------------------
  [ ] [CRITICAL] Funding rates: Binance vs Bybit correlation >0.95
  [ ] [REQUIRED] OHLCV: Cross-venue price deviation <0.5%
  [ ] [IMPORTANT] Open interest: Trend alignment across venues
  [ ] [CRITICAL] Options: Put-call parity within bounds

Outlier Detection:
------------------------------------------------------------
  [ ] [REQUIRED] Funding rate outliers flagged (|rate| > 1%)
  [ ] [REQUIRED] Price spikes >10% in 1h flagged
  [ ] [IMPORTANT] Volume anomalies detected (>5x average)
  [ ] [REQUIRED] Options IV outliers flagge

---

## 6. Risk Assessment and Mitigation

Potential risks and fallback strategies for data collection.

In [10]:
@dataclass
class Risk:
    """Risk assessment for data collection."""
    risk: str
    likelihood: str  # High, Medium, Low
    impact: str      # High, Medium, Low
    mitigation: str
    fallback: str
    
    @property
    def risk_score(self) -> int:
        score_map = {'High': 3, 'Medium': 2, 'Low': 1}
        return score_map[self.likelihood] * score_map[self.impact]

RISKS = [
    Risk(
        risk='API Rate Limit Exceeded',
        likelihood='High',
        impact='Medium',
        mitigation='Token bucket rate limiter with 20% buffer. Exponential backoff.',
        fallback='Spread collection over multiple days. Use batch endpoints.'
    ),
    Risk(
        risk='API Outage / Downtime',
        likelihood='Low',
        impact='High',
        mitigation='Checkpoint/resume capability. Automatic retry with jitter.',
        fallback='Use secondary venue (Bybit for Binance data).'
    ),
    Risk(
        risk='Data Quality Issues',
        likelihood='Medium',
        impact='Medium',
        mitigation='Real-time validation. Cross-venue correlation checks.',
        fallback='Document gaps. Exclude from analysis if unrecoverable.'
    ),
    Risk(
        risk='DEX Wash Trading',
        likelihood='High',
        impact='Medium',
        mitigation='Volume/TVL ratio filter. Transaction count thresholds.',
        fallback='Use only pools with established 30-day liquidity history.'
    ),
    Risk(
        risk='Symbol Delisting',
        likelihood='Medium',
        impact='Low',
        mitigation='Track delisting dates in metadata. Regular universe refresh.',
        fallback='Exclude from post-delist analysis. Document survivorship bias.'
    ),
    Risk(
        risk='The Graph Rate Limits',
        likelihood='Medium',
        impact='Medium',
        mitigation='Use API key (100k queries/mo). Optimize query complexity.',
        fallback='Use cached/historical snapshots. GeckoTerminal backup.'
    ),
    Risk(
        risk='Options Data Volume',
        likelihood='Medium',
        impact='Low',
        mitigation='Filter to liquid strikes only. Aggregate where possible.',
        fallback='Reduce snapshot frequency from 1h to 4h.'
    ),
    Risk(
        risk='Funding Rate Format Changes',
        likelihood='Low',
        impact='High',
        mitigation='Schema validation on every record. Version tracking.',
        fallback='Manual review and format migration.'
    ),
]

# Create risk matrix
risk_df = pd.DataFrame([
    {
        'Risk': r.risk,
        'Likelihood': r.likelihood,
        'Impact': r.impact,
        'Risk Score': r.risk_score,
        'Mitigation': r.mitigation[:50] + '...' if len(r.mitigation) > 50 else r.mitigation,
        'Fallback': r.fallback[:50] + '...' if len(r.fallback) > 50 else r.fallback
    }
    for r in RISKS
])

print("=" * 80)
print("RISK ASSESSMENT MATRIX")
print("=" * 80 + "\n")

risk_df.sort_values('Risk Score', ascending=False)

RISK ASSESSMENT MATRIX



,Risk,Likelihood,Impact,Risk Score,Mitigation,Fallback
0,API Rate Limit Exceeded,High,Medium,6,Token bucket rate limiter with 20% buffer. Exp...,Spread collection over multiple days. Use batc...
3,DEX Wash Trading,High,Medium,6,Volume/TVL ratio filter. Transaction count thr...,Use only pools with established 30-day liquidi...
2,Data Quality Issues,Medium,Medium,4,Real-time validation. Cross-venue correlation ...,Document gaps. Exclude from analysis if unreco...
5,The Graph Rate Limits,Medium,Medium,4,Use API key (100k queries/mo). Optimize query ...,Use cached/historical snapshots. GeckoTerminal...
1,API Outage / Downtime,Low,High,3,Checkpoint/resume capability. Automatic retry ...,Use secondary venue (Bybit for Binance data).
7,Funding Rate Format Changes,Low,High,3,Schema validation on every record. Version tra...,Manual review and format migration.
4,Symbol Delisting,Medium,Low,2,Track delisting dates in metadata. Regular uni...,Exclude from post-delist analysis. Document su...
6,Options Data Volume,Medium,Low,2,Filter to liquid strikes only. Aggregate where...,Reduce snapshot frequency from 1h to 4h.


---

## 7. Execution Timeline

Phased execution plan with dependencies and success criteria.

In [11]:
@dataclass
class Phase:
    """Execution phase specification."""
    phase: str
    duration: str
    tasks: List[str]
    dependencies: List[str]
    success_criteria: List[str]
    est_hours: float

PHASES = [
    Phase(
        phase='Phase 1: CEX Core Data',
        duration='Day 1-2',
        tasks=[
            'Binance funding rates (all symbols)',
            'Binance OHLCV (1h, all symbols)',
            'Binance open interest (top 20)',
            'Bybit funding rates (validation)',
        ],
        dependencies=['API credentials configured', 'Rate limiter tested'],
        success_criteria=['>97% coverage', 'Cross-validation correlation >0.95'],
        est_hours=12
    ),
    Phase(
        phase='Phase 2: Hybrid Venues',
        duration='Day 2-3',
        tasks=[
            'Hyperliquid funding + OHLCV',
            'dYdX V4 funding + OHLCV',
            'Interval normalization (1h → 8h)',
        ],
        dependencies=['Phase 1 complete'],
        success_criteria=['>95% coverage', 'Normalized rates align'],
        est_hours=8
    ),
    Phase(
        phase='Phase 3: DEX Data',
        duration='Day 3-4',
        tasks=[
            'Uniswap V3 pools (Ethereum + Arbitrum)',
            'Pool TVL and swap history',
            'Wash trading filter application',
        ],
        dependencies=['The Graph API key'],
        success_criteria=['TVL filter applied', 'Wash trading flagged'],
        est_hours=6
    ),
    Phase(
        phase='Phase 4: Options & Futures',
        duration='Day 4-5',
        tasks=[
            'Deribit full options chain',
            'BTC futures term structure',
            'DVOL index',
        ],
        dependencies=['Deribit API configured'],
        success_criteria=['All expiries captured', 'Greeks validated'],
        est_hours=16
    ),
    Phase(
        phase='Phase 5: Validation & QA',
        duration='Day 5-6',
        tasks=[
            'Cross-venue validation',
            'Outlier detection',
            'Gap analysis',
            'Quality score calculation',
        ],
        dependencies=['All data collected'],
        success_criteria=['Quality score >90/100', 'All critical checks pass'],
        est_hours=8
    ),
    Phase(
        phase='Phase 6: Documentation',
        duration='Day 6-7',
        tasks=[
            'Data dictionary',
            'Source attribution',
            'Quality report',
            'Limitations documentation',
        ],
        dependencies=['QA complete'],
        success_criteria=['All docs complete', 'Reproducible collection'],
        est_hours=4
    ),
]

# Create timeline
timeline_df = pd.DataFrame([
    {
        'Phase': p.phase,
        'Duration': p.duration,
        'Est. Hours': p.est_hours,
        'Tasks': len(p.tasks),
        'Key Deliverable': p.tasks[0],
    }
    for p in PHASES
])

print("=" * 80)
print("EXECUTION TIMELINE")
print("=" * 80 + "\n")

display(timeline_df)

total_hours = sum(p.est_hours for p in PHASES)
print(f"\nTOTAL ESTIMATED TIME: {total_hours} hours ({total_hours/8:.1f} working days)")

EXECUTION TIMELINE



,Phase,Duration,Est. Hours,Tasks,Key Deliverable
0,Phase 1: CEX Core Data,Day 1-2,12,4,Binance funding rates (all symbols)
1,Phase 2: Hybrid Venues,Day 2-3,8,3,Hyperliquid funding + OHLCV
2,Phase 3: DEX Data,Day 3-4,6,3,Uniswap V3 pools (Ethereum + Arbitrum)
3,Phase 4: Options & Futures,Day 4-5,16,3,Deribit full options chain
4,Phase 5: Validation & QA,Day 5-6,8,4,Cross-venue validation
5,Phase 6: Documentation,Day 6-7,4,4,Data dictionary



TOTAL ESTIMATED TIME: 54 hours (6.8 working days)


In [12]:
# Detailed phase breakdown
print("=" * 80)
print("DETAILED PHASE BREAKDOWN")
print("=" * 80)

for phase in PHASES:
    print(f"\n{phase.phase}")
    print(f"Duration: {phase.duration} | Est: {phase.est_hours}h")
    print("-" * 60)
    
    print("Tasks:")
    for task in phase.tasks:
        print(f"  - {task}")
    
    print("Dependencies:")
    for dep in phase.dependencies:
        print(f"  > {dep}")
    
    print("Success Criteria:")
    for criteria in phase.success_criteria:
        print(f"  [OK] {criteria}")

DETAILED PHASE BREAKDOWN

Phase 1: CEX Core Data
Duration: Day 1-2 | Est: 12h
------------------------------------------------------------
Tasks:
  - Binance funding rates (all symbols)
  - Binance OHLCV (1h, all symbols)
  - Binance open interest (top 20)
  - Bybit funding rates (validation)
Dependencies:
  > API credentials configured
  > Rate limiter tested
Success Criteria:
  [OK] >97% coverage
  [OK] Cross-validation correlation >0.95

Phase 2: Hybrid Venues
Duration: Day 2-3 | Est: 8h
------------------------------------------------------------
Tasks:
  - Hyperliquid funding + OHLCV
  - dYdX V4 funding + OHLCV
  - Interval normalization (1h → 8h)
Dependencies:
  > Phase 1 complete
Success Criteria:
  [OK] >95% coverage
  [OK] Normalized rates align

Phase 3: DEX Data
Duration: Day 3-4 | Est: 6h
------------------------------------------------------------
Tasks:
  - Uniswap V3 pools (Ethereum + Arbitrum)
  - Pool TVL and swap history
  - Wash trading filter application
Dependencie

---

## Summary

### Key Decisions

| Decision | Rationale |
|----------|----------|
| **Binance as primary CEX** | Best coverage, highest rate limits, 8h funding |
| **Hyperliquid as primary hybrid** | Best liquidity, zero maker fees, 1h funding |
| **Cross-validate with Bybit** | Independent data quality verification |
| **Aggressive DEX filtering** | Mitigate wash trading, ensure liquidity |
| **Parquet with monthly partitions** | Efficient access, good compression |

### Data Sources Summary

| Type | Primary | Validation | Data Types |
|------|---------|------------|------------|
| CEX | Binance | Bybit | Funding, OHLCV, OI |
| Hybrid | Hyperliquid | dYdX V4 | Funding, OHLCV |
| DEX | Uniswap V3 | GeckoTerminal | Pools, TVL |
| Options | Deribit | - | Chain, DVOL |

### Timeline Summary

- **Total Duration**: 7 days
- **Estimated Hours**: 54 hours of execution
- **Storage Required**: ~6 GB compressed
- **Quality Target**: >95% coverage, <5% missing